# QuantJourney SDK - Macro Rates Equity Factor Panel

This notebook demonstrates a QuantJourney SDK workflow that links inflation, labor, rates, breakevens, Fama-French factors and equity sector returns into a regime-conditioned factor panel.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
def series_from_payload(name: str, payload: Any) -> pd.Series:
    frame = pd.DataFrame(as_rows(payload))
    if frame.empty:
        return pd.Series(dtype=float, name=name)
    date_col = next((col for col in frame.columns if 'date' in str(col).lower() or str(col).lower() in {'observation_date', 'time'}), frame.columns[0])
    numeric_cols = frame.select_dtypes(include='number').columns.tolist()
    value_col = 'value' if 'value' in frame.columns else 'close' if 'close' in frame.columns else numeric_cols[-1] if numeric_cols else None
    if value_col is None:
        return pd.Series(dtype=float, name=name)
    frame['date'] = pd.to_datetime(frame[date_col], errors='coerce')
    frame[name] = pd.to_numeric(frame[value_col], errors='coerce')
    return frame.dropna(subset=['date', name]).set_index('date')[name].sort_index()


In [ ]:
sector_etfs = ['XLK', 'XLF', 'XLE', 'XLV', 'XLY', 'XLP', 'XLU', 'XLI']
macro_raw = {'CPI Urban Consumers YoY (FRED:CPIAUCSL)': qj.fred.get_cpi(start_date='2018-01-01'), 'Civilian Unemployment Rate (FRED:UNRATE)': qj.fred.get_fred_data_series_by_id(series_id='UNRATE', start='2018-01-01'), '10Y Breakeven Inflation (FRED:T10YIE)': qj.fred.get_fred_data_series_by_id(series_id='T10YIE', start='2018-01-01'), 'Effective Fed Funds Rate (FRED:FEDFUNDS)': qj.fred.get_effective_federal_funds_rate(start_date='2018-01-01'), '2-Year Treasury Yield (FRED:DGS2)': qj.fred.get_treasury_2y(start_date='2018-01-01'), '10-Year Treasury Yield (FRED:DGS10)': qj.fred.get_treasury_10y(start_date='2018-01-01')}
ff_raw = qj.ff.get_factors(region='US')
prices, volumes = price_panel(sector_etfs + ['SPY', 'TLT'], start='2018-01-01', end=END)


In [ ]:
macro = pd.concat([series_from_payload(name, payload) for name, payload in macro_raw.items()], axis=1).dropna(how='all')
ret = returns(prices).dropna()
factor_rows = pd.DataFrame(as_rows(ff_raw))
if not factor_rows.empty:
    date_col = next((col for col in factor_rows.columns if 'date' in str(col).lower()), factor_rows.columns[0])
    factor_rows['date'] = pd.to_datetime(factor_rows[date_col], errors='coerce')
    factor_rows = factor_rows.dropna(subset=['date']).set_index('date').sort_index()


In [ ]:
rates_proxy = ret['TLT'].rolling(63).sum() * -1
inflation_proxy = ret['XLE'].rolling(63).sum() - ret['TLT'].rolling(63).sum()
regime = pd.DataFrame({'rising_rate_regime': rates_proxy > rates_proxy.rolling(252).median(), 'inflation_pressure_regime': inflation_proxy > inflation_proxy.rolling(252).median()}, index=ret.index).dropna()
sector_returns = ret[sector_etfs].join(regime).dropna()
conditioned = sector_returns.groupby(['rising_rate_regime', 'inflation_pressure_regime'])[sector_etfs].mean() * 252
display(macro.tail())
display(factor_rows.tail() if not factor_rows.empty else pd.DataFrame())
display(conditioned)
conditioned.T.plot(kind='bar', title='Sector returns by rates and inflation regime')
plt.ylabel('annualized return')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.